# Domain-Specific Safety Reviews
Companion notebook for the **Domain-Specific Safety Reviews** section of Chapter 11.

Review safety by **domain** and by **task type** -- a low-risk, read-only lookup needs one level of control; a task that changes records, moves money, updates schedules, or affects care needs a much stricter one.

This is the book's `safety_ready` release checklist, unchanged:

In [1]:
def safety_ready(review):
    return all([
        review["access_checked"],
        review["consent_checked"],
        review["approval_gate_defined"],
        review["abuse_tests_passed"],
        review["audit_logging_on"],
        review["fallback_defined"],
        review["monitoring_enabled"],
    ])

In [2]:
complete_review = {
    "access_checked": True, "consent_checked": True, "approval_gate_defined": True,
    "abuse_tests_passed": True, "audit_logging_on": True, "fallback_defined": True,
    "monitoring_enabled": True,
}
incomplete_review = {**complete_review, "abuse_tests_passed": False}

print("complete review ready to launch:  ", safety_ready(complete_review))
print("incomplete review ready to launch:", safety_ready(incomplete_review))

complete review ready to launch:   True
incomplete review ready to launch: False


A checklist that only says pass/fail isn't very actionable -- a small helper that reports exactly what's missing makes the review usable in practice:

In [3]:
def missing_review_items(review: dict) -> list[str]:
    required = [
        "access_checked", "consent_checked", "approval_gate_defined",
        "abuse_tests_passed", "audit_logging_on", "fallback_defined", "monitoring_enabled",
    ]
    return [item for item in required if not review.get(item)]


print("missing before launch:", missing_review_items(incomplete_review))

missing before launch: ['abuse_tests_passed']


The four review dimensions the book calls out -- **data privacy**, **action approval**, **bias and fairness** (wherever ranking, triage, or prioritization is involved), and **traceability** -- map directly onto those checklist fields: `access_checked` covers privacy, `approval_gate_defined` covers approval, `audit_logging_on` covers traceability, and `abuse_tests_passed` is where you'd test stale data, prompt injection, and fairness across groups before launch.

Safety review doesn't stop at launch -- track the same signals in production:

In [4]:
def post_launch_alert(unsafe_action_attempts: int, escalation_rate: float, policy_violations: int) -> bool:
    return unsafe_action_attempts > 0 or escalation_rate > 0.25 or policy_violations > 0


print(post_launch_alert(unsafe_action_attempts=0, escalation_rate=0.08, policy_violations=0))
print(post_launch_alert(unsafe_action_attempts=1, escalation_rate=0.08, policy_violations=0))

False
True
